# turkish_distill: contrastive baseline

Thin notebook. All logic lives in `src/turkish_distill`. Trains the student on (question, passage) pairs with an in-batch-negative loss and no teachers; this is the non-distillation baseline next to the single- and two-teacher runs. Run the cells top to bottom; training resumes by itself when `RUN_ID` points to an existing run.

Needs `data/stage1/pairs_train.jsonl` and `pairs_val.jsonl` on Drive (made locally with `prepare_pairs`, then uploaded). Model selection uses the validation retrieval metrics (`val/*`); TR-MTEB is run separately, once, on the chosen models.

## 1. Setup (Drive, repo, install)

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # only needed for gated models
except Exception as exc:
    print('HF_TOKEN not set:', exc)

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

## 2. Choose the experiment

Everything below is a parameter; the run folder name carries the settings, so runs never overwrite each other.

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive

# --- Experiment choice ---
CONFIG = 'configs/contrastive_mnrl.yaml'
STUDENT = 'ytu-ce-cosmos/modernbert-tr-base-1k'
STUDENT_REVISION = 'c2e35f553101cce5b03fbb318ef8728473fd34dd'  # HF commit hash of that model; None = latest
LOSS = 'mnrl'        # 'mnrl' (question -> passage) or 'mnrl_symmetric' (both directions)
SCALE = 20.0         # 1 / temperature (the lab's 0.02 is 50)
BATCH_SIZE = 128     # pairs per batch; every other passage in the batch is a negative
CHUNK_SIZE = 32      # gradient caching: same maths, memory of 32 pairs. None ran out of memory on an L4 at batch 128; BATCH_SIZE can go to 1024+ at the same memory
LR = 3e-5
EPOCHS = 3
# Stage-1 contrastive grid (one change from the baseline LOSS='mnrl', SCALE=20, LR=3e-5, EPOCHS=3; TR-MTEB 59.62):
#   run 1: SCALE = 50                         (temperature 0.02, the lab's value)
#   run 2: LOSS = 'mnrl_symmetric'            (both directions, scale 20)
#   run 3: LR = 5e-5, EPOCHS = 6              (closer to the distillation budget of 1800 steps)
MAX_STEPS = None     # set for a quick trial; takes precedence over EPOCHS
MAX_TRAIN_PAIRS = None   # e.g. 2000 for a quick trial
RUN_ID = None        # set e.g. '20260101-120000-abc1234' to resume an existing run; None starts a new run

NAME = f"c_{STUDENT.split('/')[-1]}_{LOSS}_s{SCALE:g}_b{BATCH_SIZE}" + (f'_chunk{CHUNK_SIZE}' if CHUNK_SIZE else '')
if (LR, EPOCHS) != (3e-5, 3):   # the baseline name stays as it was; other schedules are told apart in the folder name
    NAME += f'_lr{LR:g}_ep{EPOCHS}'


def ov(key, value):
    return f'{key}={"null" if value is None else value}'


OVERRIDES = [
    ov('name', NAME), ov('student.hf_id', STUDENT), ov('student.revision', STUDENT_REVISION),
    ov('loss.name', LOSS), ov('loss.scale', SCALE),
    ov('train.batch_size', BATCH_SIZE), ov('train.chunk_size', CHUNK_SIZE), ov('train.lr', LR),
    ov('train.epochs', EPOCHS), ov('train.max_steps', MAX_STEPS), ov('data.max_train_pairs', MAX_TRAIN_PAIRS),
]
override_args = ' '.join(f'--override {o}' for o in OVERRIDES)

PAIRS_DIR = f'{DRIVE_ROOT}/data/stage1'
missing = [f for f in ('pairs_train.jsonl', 'pairs_val.jsonl') if not os.path.exists(f'{PAIRS_DIR}/{f}')]
if missing:
    raise FileNotFoundError(f'{missing} not found in {PAIRS_DIR}: upload the output of prepare_pairs first.')
print(NAME, RUN_ID)

## 3. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/bitirme/runs

## 4. Train

Logs `val/mrr@10`, `val/recall@1`, `val/recall@10` at step 0 (the raw student) and every `eval_every` steps.

In [ ]:
run_id_arg = f'--run-id {RUN_ID}' if RUN_ID else ''
!python -m turkish_distill.train_contrastive --config {CONFIG} {run_id_arg} {override_args}

## 5. Compare the contrastive runs

The last validation row of every run (start vs end), best first. The validation set is made of pairs the training never saw; it is the selection signal, not a final result.

In [ ]:
import glob, yaml, pandas as pd
from turkish_distill.logging_utils import read_metrics

rows = []
for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/c_*/*')):
    try:
        cfg = yaml.safe_load(open(f'{d}/config.yaml'))
        val = [r for r in read_metrics(f'{d}/metrics.jsonl') if 'val/mrr@10' in r]
    except Exception:
        continue
    if not val:
        continue
    rows.append({
        'run': d.split('/runs/')[1], 'loss': cfg['loss']['name'], 'scale': cfg['loss']['scale'],
        'batch': cfg['train']['batch_size'], 'chunk': cfg['train']['chunk_size'], 'lr': cfg['train']['lr'],
        'step': val[-1]['step'], 'mrr@10 start': val[0]['val/mrr@10'], 'mrr@10': val[-1]['val/mrr@10'],
        'recall@1': val[-1]['val/recall@1'], 'recall@10': val[-1]['val/recall@10'],
    })
pd.DataFrame(rows).sort_values('mrr@10', ascending=False).round(4) if rows else print('no finished validation rows yet')

## 6. After training

The trained backbone is in `runs/<name>/<run_id>/final/` (HF format, mean pooling). To score it on TR-MTEB, open `colab_eval.ipynb` and evaluate that folder with the same protocol as the other models. Do that only for the runs chosen on the validation metrics above.